# Experiment 1.3.6 — SNN representation and readout ablation

## Motivation

Experiment 1.3.5 showed that the continuous SNN can fit the training data and learn discriminative representations, but the final **SNN + 250 ms output spike-count + Linear** system generalizes worse to held-out test users than the direct **250 ms input count + Linear** baseline.

This notebook does **not** retrain the SNN backbone. Instead, it reuses the three completed Experiment 1.3.5 checkpoints (`seed 11 / 23 / 101`), freezes each SNN, extracts intermediate representations, and trains only a fresh linear probe.

The goal is to localize where information/generalization is being lost.

### A — Layer probe

With the same 250 ms count readout:

\[
\text{Raw input count}
\quad vs\quad
\text{L1 spike count}
\quad vs\quad
\text{L2 spike count}
\quad vs\quad
\text{L3 spike count}
\]

This asks whether deeper SNN transformations progressively improve or degrade cross-user representations.

### B — Temporal-resolution probe

Keep the frozen L3 spike train fixed, but retain progressively more output timing:

\[
250\text{ ms} \; (16\text{ samples})
\quad vs\quad
125\text{ ms} \; (8\text{ samples})
\quad vs\quad
62.5\text{ ms} \; (4\text{ samples})
\]

If finer L3 readout improves performance, the SNN may be producing useful temporal codes that were lost by the 250 ms output-count compression.

### C — State-readout probe

Compare different frozen L3 representations over the same 250 ms coarse bins:

\[
\text{L3 spike count}
\quad vs\quad
\text{L3 membrane end-state}
\quad vs\quad
\text{L3 membrane mean}
\]

This asks whether information is present in the continuous neuron state but is lost by thresholding and spike-count readout.

## Important controls

- same 64 Hz / 30-channel unsigned event cohort as Experiment 1.3.5;
- same fixed user-disjoint split (`SPLIT_SEED=12345`);
- same frozen best checkpoints from Experiment 1.3.5;
- no end-to-end SNN retraining;
- every probe uses **train-only `StandardScaler` + fresh `LogisticRegression`**;
- test data are never used for model fitting or model selection.

In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import math
import os
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import snntorch as snn
from snntorch import surrogate

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the writingRing repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from snn.accel_reconstruction_eval.datasets import load_acceleration_data

print("Repository root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Repository root: /home/ted/project/writingRing
PyTorch: 2.11.0+cu128
CUDA available: True


## 1. Configuration

In [2]:
# Dataset: match Experiment 1.3.5 exactly
DATASET_ROOTS = [
    REPO_ROOT / "outputs/action0_wavelets_0e5_1_2_4_8_sr_64/low-pass/aligned-board-events/segmentation_padded",
    REPO_ROOT / "outputs/action1_wavelets_0e5_1_2_4_8_sr_64/low-pass/aligned-board-events/segmentation_padded",
]

EXPECTED_EVENT_REPRESENTATION = "unsigned"
EXPECTED_EVENT_FEATURE_SCHEMA = "custom_wavelet_polarity_split_abs_events_v1"
EVENT_CHANNEL_COUNT = 30
TOTAL_CHANNEL_COUNT = 36
EXPECTED_SAMPLING_RATE_HZ = 64.0

INCLUDED_LABELS = (
    "A", "B", "C", "D", "E", "X",
    "G", "H", "I", "J", "K", "L",
)

SPLIT_SEED = 12345
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
SNN_SEEDS = (11, 23, 101)

# Must match the trained 1.3.5 backbone
SNN_LAYER_WIDTHS = (128, 128, 64)
SNN_LAYER_SHIFTS = (
    (2, 3),
    (2, 3),
    (2, 3),
)
TAU_MEM_MS = 22.54
THRESHOLD = 0.5
SURROGATE_SLOPE = 25.0
RESET_MECHANISM = "subtract"

COARSE_BIN_MS = 250.0
TEMPORAL_READOUT_SAMPLES = (16, 8, 4)  # 250 / 125 / 62.5 ms at 64 Hz

BATCH_SIZE = 64
NUM_WORKERS = 0
LOGREG_MAX_ITER = 5000
LOGREG_C = 1.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SOURCE_EXPERIMENT_ID = "experiment_1_3_5_continuous_snn_local_features"
SOURCE_RESULTS_DIR = REPO_ROOT / "notebooks" / "artifacts" / SOURCE_EXPERIMENT_ID

EXPERIMENT_ID = "experiment_1_3_6_snn_representation_readout_ablation"
RESULTS_DIR = REPO_ROOT / "notebooks" / "artifacts" / EXPERIMENT_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Source checkpoints:", SOURCE_RESULTS_DIR)
print("Output dir:", RESULTS_DIR)
print("Device:", DEVICE)

Source checkpoints: /home/ted/project/writingRing/notebooks/artifacts/experiment_1_3_5_continuous_snn_local_features
Output dir: /home/ted/project/writingRing/notebooks/artifacts/experiment_1_3_6_snn_representation_readout_ablation
Device: cuda


## 2. Reproducibility and SNN helper functions

In [3]:
def derive_seed(master_seed: int, *parts: object) -> int:
    text = "|".join([str(master_seed), *(str(p) for p in parts)])
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little", signed=False)


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def shift_to_alpha(shift: int) -> float:
    return float(1.0 - 2.0 ** (-int(shift)))


def allocate_neurons(width: int, shifts: tuple[int, ...]) -> tuple[int, ...]:
    base, remainder = divmod(int(width), len(shifts))
    counts = [base] * len(shifts)
    order = []
    left, right = 0, len(shifts) - 1
    while left <= right:
        order.append(left)
        if right != left:
            order.append(right)
        left += 1
        right -= 1
    for i in range(remainder):
        counts[order[i]] += 1
    return tuple(counts)


def build_alpha_tensor(width: int, shifts: tuple[int, ...]) -> torch.Tensor:
    vals = []
    for shift, count in zip(shifts, allocate_neurons(width, shifts), strict=True):
        vals.extend([shift_to_alpha(shift)] * count)
    if len(vals) != width:
        raise AssertionError((len(vals), width))
    return torch.tensor(vals, dtype=torch.float32)


seed_everything(2026)

## 3. Load and validate the same cohort used by Experiment 1.3.5

In [4]:
data = load_acceleration_data(
    DATASET_ROOTS,
    repository_root=REPO_ROOT,
    require_reconstruction=False,
)

sampling_rates = {float(m.sampling_rate_hz) for m in data.producer_metadatas}
if len(sampling_rates) != 1:
    raise ValueError(f"Expected one shared sampling rate, got {sampling_rates}")
SAMPLING_RATE_HZ = sampling_rates.pop()

if not np.isclose(SAMPLING_RATE_HZ, EXPECTED_SAMPLING_RATE_HZ):
    raise ValueError((SAMPLING_RATE_HZ, EXPECTED_SAMPLING_RATE_HZ))

for root, metadata in zip(data.padded_roots, data.producer_metadatas, strict=True):
    raw = metadata.raw
    if raw.get("event_representation") != EXPECTED_EVENT_REPRESENTATION:
        raise ValueError(f"{root}: wrong event_representation")
    if raw.get("event_feature_schema") != EXPECTED_EVENT_FEATURE_SCHEMA:
        raise ValueError(f"{root}: wrong event_feature_schema")
    if raw.get("event_channel_count") != EVENT_CHANNEL_COUNT:
        raise ValueError(f"{root}: expected 30 event channels")
    if metadata.channel_count != TOTAL_CHANNEL_COUNT:
        raise ValueError(f"{root}: expected 36 total channels")

rows = []
keep = set(INCLUDED_LABELS)
for package_index, package in enumerate(data.packages):
    for segment_index, label in enumerate(package.labels.astype(str)):
        if label not in keep:
            continue
        rows.append({
            "package_index": package_index,
            "segment_index": segment_index,
            "user": str(package.user),
            "action": str(package.action),
            "label": str(label),
            "valid_length": int(package.valid_lengths[segment_index]),
            "package_padded_length": int(package.padded_spike_imu.shape[1]),
            "sample_id": f"{package.user}/action_{package.action}/{segment_index}",
        })

manifest = pd.DataFrame(rows)
labels_sorted = sorted(manifest.label.unique().tolist())
CLASS_TO_IDX = {lab: i for i, lab in enumerate(labels_sorted)}
manifest["label_idx"] = manifest.label.map(CLASS_TO_IDX).astype(int)
N_CLASSES = len(labels_sorted)
GLOBAL_PADDED_LENGTH = int(manifest.package_padded_length.max())

COARSE_BIN_SAMPLES = int(np.rint(COARSE_BIN_MS * SAMPLING_RATE_HZ / 1000.0))
if COARSE_BIN_SAMPLES != 16:
    raise ValueError(f"Expected 16 samples for 250 ms at 64 Hz, got {COARSE_BIN_SAMPLES}")

N_COARSE_BINS = int(math.ceil(GLOBAL_PADDED_LENGTH / COARSE_BIN_SAMPLES))
PADDED_LENGTH = N_COARSE_BINS * COARSE_BIN_SAMPLES

print(f"samples={len(manifest)}, users={manifest.user.nunique()}, classes={N_CLASSES}")
print(f"sampling_rate={SAMPLING_RATE_HZ} Hz")
print(f"global padded length={GLOBAL_PADDED_LENGTH}; analysis padded length={PADDED_LENGTH}")
print(f"coarse readout: {COARSE_BIN_SAMPLES} samples = {COARSE_BIN_MS} ms; bins={N_COARSE_BINS}")
display(manifest.head())

samples=853, users=20, classes=12
sampling_rate=64.0 Hz
global padded length=256; analysis padded length=256
coarse readout: 16 samples = 250.0 ms; bins=16


,package_index,segment_index,user,action,label,valid_length,package_padded_length,sample_id,label_idx
0,0,24,user_0,0,J,72,256,user_0/action_0/24,8
1,0,25,user_0,0,B,128,256,user_0/action_0/25,1
2,0,26,user_0,0,X,106,256,user_0/action_0/26,11
3,0,27,user_0,0,E,122,256,user_0/action_0/27,4
4,0,31,user_0,0,G,124,256,user_0/action_0/31,5


## 4. Reconstruct the exact fixed user-disjoint split

In [5]:
def check_event_padding_is_zero(tol: float = 1e-12):
    bad = []
    for row in manifest.itertuples(index=False):
        package = data.packages[int(row.package_index)]
        tail = np.asarray(package.padded_spike_imu[
            int(row.segment_index), int(row.valid_length):, :EVENT_CHANNEL_COUNT
        ])
        mx = float(np.max(np.abs(tail))) if tail.size else 0.0
        if mx > tol:
            bad.append((row.sample_id, mx))
    if bad:
        raise ValueError(f"Non-zero event padding detected: {bad[:5]}")
    print("Padding contract verified: padded event tails are zero.")


def make_user_split(split_seed: int):
    users = np.asarray(sorted(manifest.user.unique().tolist()), dtype=object)
    rng = np.random.default_rng(derive_seed(split_seed, "user_split"))
    rng.shuffle(users)

    n_users = len(users)
    n_train = int(round(TRAIN_FRACTION * n_users))
    n_val = int(round(VAL_FRACTION * n_users))

    train_users = set(users[:n_train])
    val_users = set(users[n_train:n_train+n_val])
    test_users = set(users[n_train+n_val:])

    def part(us):
        return manifest[manifest.user.isin(us)].reset_index(drop=True)

    return part(train_users), part(val_users), part(test_users), {
        "train_users": tuple(sorted(train_users)),
        "val_users": tuple(sorted(val_users)),
        "test_users": tuple(sorted(test_users)),
    }


check_event_padding_is_zero()
train_df, val_df, test_df, split_info = make_user_split(SPLIT_SEED)

display(pd.DataFrame([
    {"split":"train", "samples":len(train_df), "users":train_df.user.nunique(), "user_ids":",".join(split_info["train_users"])},
    {"split":"val", "samples":len(val_df), "users":val_df.user.nunique(), "user_ids":",".join(split_info["val_users"])},
    {"split":"test", "samples":len(test_df), "users":test_df.user.nunique(), "user_ids":",".join(split_info["test_users"])},
]))

Padding contract verified: padded event tails are zero.


,split,samples,users,user_ids
0,train,632,14,"user_0,user_1,user_11,user_12,user_14,user_19,..."
1,val,126,3,"user_15,user_16,user_20"
2,test,95,3,"user_10,user_13,user_18"


## 5. Build the common continuous input tensors

In [6]:
def build_global_padded_events(df: pd.DataFrame) -> np.ndarray:
    out = np.zeros((len(df), PADDED_LENGTH, EVENT_CHANNEL_COUNT), dtype=np.float32)
    for i, row in enumerate(df.itertuples(index=False)):
        package = data.packages[int(row.package_index)]
        x = np.asarray(
            package.padded_spike_imu[int(row.segment_index), :, :EVENT_CHANNEL_COUNT],
            dtype=np.float32,
        )
        n = min(len(x), PADDED_LENGTH)
        out[i, :n] = x[:n]
    return out


def labels_for_df(df: pd.DataFrame) -> np.ndarray:
    return df.label_idx.to_numpy(dtype=np.int64)


X_train = build_global_padded_events(train_df)
X_val = build_global_padded_events(val_df)
X_test = build_global_padded_events(test_df)
y_train = labels_for_df(train_df)
y_val = labels_for_df(val_df)
y_test = labels_for_df(test_df)

print("train:", X_train.shape, y_train.shape)
print("val:  ", X_val.shape, y_val.shape)
print("test: ", X_test.shape, y_test.shape)

train: (632, 256, 30) (632,)
val:   (126, 256, 30) (126,)
test:  (95, 256, 30) (95,)


## 6. Define the exact Experiment 1.3.5 model for checkpoint restoration

In [ ]:
# =====================================================================
# Load the actual Experiment 1.3.5 architecture from checkpoints
# =====================================================================

def checkpoint_path(seed: int) -> Path:
    return SOURCE_RESULTS_DIR / f"snn_seed_{seed}.pt"


def canonical_checkpoint_config(payload: dict) -> dict:
    """
    Normalize the Experiment 1.3.5 checkpoint config into one canonical form.

    The actual 1.3.5 checkpoint uses:
        fs
        bin_samples
        widths
        shifts
        tau_mem_ms
        threshold
        split_seed
    """
    cfg = payload.get("config")
    if cfg is None:
        raise ValueError("Checkpoint does not contain a 'config' field")

    required = (
        "fs",
        "bin_samples",
        "widths",
        "shifts",
        "tau_mem_ms",
        "threshold",
        "split_seed",
    )

    missing = [key for key in required if key not in cfg]
    if missing:
        raise ValueError(
            f"Checkpoint config is missing required fields: {missing}"
        )

    return {
        "fs": float(cfg["fs"]),
        "bin_samples": int(cfg["bin_samples"]),
        "widths": tuple(int(v) for v in cfg["widths"]),
        "shifts": tuple(
            tuple(int(s) for s in layer_shifts)
            for layer_shifts in cfg["shifts"]
        ),
        "tau_mem_ms": float(cfg["tau_mem_ms"]),
        "threshold": float(cfg["threshold"]),
        "split_seed": int(cfg["split_seed"]),
    }


# ---------------------------------------------------------------------
# Seed 11 defines the actual architecture used in Experiment 1.3.5.
# ---------------------------------------------------------------------

reference_path = checkpoint_path(SNN_SEEDS[0])

if not reference_path.exists():
    raise FileNotFoundError(
        f"Missing Experiment 1.3.5 checkpoint: {reference_path}"
    )

reference_payload = torch.load(
    reference_path,
    map_location="cpu",
)

REFERENCE_CONFIG = canonical_checkpoint_config(reference_payload)

print("Actual Experiment 1.3.5 checkpoint config:")
print(REFERENCE_CONFIG)


# ---------------------------------------------------------------------
# Adopt checkpoint config as source of truth.
# ---------------------------------------------------------------------

CHECKPOINT_FS = REFERENCE_CONFIG["fs"]
CHECKPOINT_BIN_SAMPLES = REFERENCE_CONFIG["bin_samples"]
SNN_LAYER_WIDTHS = REFERENCE_CONFIG["widths"]
SNN_LAYER_SHIFTS = REFERENCE_CONFIG["shifts"]
TAU_MEM_MS = REFERENCE_CONFIG["tau_mem_ms"]
THRESHOLD = REFERENCE_CONFIG["threshold"]
CHECKPOINT_SPLIT_SEED = REFERENCE_CONFIG["split_seed"]

# Dataset-level sanity checks.
if not np.isclose(CHECKPOINT_FS, SAMPLING_RATE_HZ):
    raise ValueError(
        f"Sampling-rate mismatch: checkpoint={CHECKPOINT_FS}, "
        f"dataset={SAMPLING_RATE_HZ}"
    )

if CHECKPOINT_BIN_SAMPLES != COARSE_BIN_SAMPLES:
    raise ValueError(
        f"Bin-size mismatch: checkpoint={CHECKPOINT_BIN_SAMPLES}, "
        f"notebook={COARSE_BIN_SAMPLES}"
    )

if CHECKPOINT_SPLIT_SEED != SPLIT_SEED:
    raise ValueError(
        f"Split-seed mismatch: checkpoint={CHECKPOINT_SPLIT_SEED}, "
        f"notebook={SPLIT_SEED}"
    )

if len(SNN_LAYER_WIDTHS) != 3:
    raise ValueError(
        f"Expected 3 SNN layers, got widths={SNN_LAYER_WIDTHS}"
    )

if len(SNN_LAYER_SHIFTS) != 3:
    raise ValueError(
        f"Expected 3 shift configurations, got shifts={SNN_LAYER_SHIFTS}"
    )


print()
print("Frozen 1.3.5 architecture:")
print("  widths:", SNN_LAYER_WIDTHS)
print("  shifts:", SNN_LAYER_SHIFTS)
print("  tau_mem_ms:", TAU_MEM_MS)
print("  threshold:", THRESHOLD)


# ---------------------------------------------------------------------
# Reconstruct the exact 1.3.5 model architecture.
# ---------------------------------------------------------------------

BETA = float(
    math.exp(
        -(1000.0 / SAMPLING_RATE_HZ) / TAU_MEM_MS
    )
)


class ContinuousLocalFeatureSNN(nn.Module):
    def __init__(self):
        super().__init__()

        h1, h2, d = SNN_LAYER_WIDTHS

        spike_grad = surrogate.fast_sigmoid(
            slope=SURROGATE_SLOPE
        )

        self.fc1 = nn.Linear(
            EVENT_CHANNEL_COUNT,
            h1,
            bias=False,
        )

        self.lif1 = snn.Synaptic(
            alpha=build_alpha_tensor(
                h1,
                SNN_LAYER_SHIFTS[0],
            ),
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            reset_mechanism=RESET_MECHANISM,
        )

        self.fc2 = nn.Linear(
            h1,
            h2,
            bias=False,
        )

        self.lif2 = snn.Synaptic(
            alpha=build_alpha_tensor(
                h2,
                SNN_LAYER_SHIFTS[1],
            ),
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            reset_mechanism=RESET_MECHANISM,
        )

        self.fc3 = nn.Linear(
            h2,
            d,
            bias=False,
        )

        self.lif3 = snn.Synaptic(
            alpha=build_alpha_tensor(
                d,
                SNN_LAYER_SHIFTS[2],
            ),
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            reset_mechanism=RESET_MECHANISM,
        )

        self.classifier = nn.Linear(
            N_COARSE_BINS * d,
            N_CLASSES,
            bias=True,
        )


# ---------------------------------------------------------------------
# Load each frozen checkpoint.
# Only require all SNN seeds to use the same actual architecture.
# ---------------------------------------------------------------------

def load_frozen_checkpoint(seed: int):
    path = checkpoint_path(seed)

    if not path.exists():
        raise FileNotFoundError(
            f"Missing Experiment 1.3.5 checkpoint: {path}"
        )

    payload = torch.load(
        path,
        map_location="cpu",
    )

    got = canonical_checkpoint_config(payload)

    if got != REFERENCE_CONFIG:
        print("Reference config:", REFERENCE_CONFIG)
        print("Checkpoint config:", got)
        raise ValueError(
            f"Checkpoint architecture/config mismatch for seed {seed}"
        )

    model = ContinuousLocalFeatureSNN()

    model.load_state_dict(
        payload["model_state_dict"],
        strict=True,
    )

    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad_(False)

    return model, payload


# ---------------------------------------------------------------------
# Verify all requested 1.3.5 checkpoints.
# ---------------------------------------------------------------------

checkpoint_rows = []

for seed in SNN_SEEDS:
    model, payload = load_frozen_checkpoint(seed)

    checkpoint_rows.append(
        {
            "seed": seed,
            "best_epoch": payload.get("best_epoch"),
            "val_BA": payload["val"]["balanced_accuracy"],
            "test_BA_original_1_3_5": payload["test"][
                "balanced_accuracy"
            ],
            "widths": str(SNN_LAYER_WIDTHS),
            "shifts": str(SNN_LAYER_SHIFTS),
        }
    )

display(pd.DataFrame(checkpoint_rows))

Expected config: {'experiment_id': 'experiment_1_3_5_continuous_snn_local_features', 'sampling_rate_hz': 64.0, 'event_channels': 30, 'global_padded_length': 256, 'padded_readout_length': 256, 'bin_samples': 16, 'fixed_bin_ms': 250.0, 'layer_widths': (128, 128, 64), 'layer_shifts': ((2, 3), (2, 3), (2, 3)), 'tau_mem_ms': 22.54, 'threshold': 0.5, 'split_seed': 12345}
Checkpoint config: {'experiment_id': 'experiment_1_3_5_continuous_snn_local_features', 'fs': 64.0, 'bin_samples': 16, 'widths': (128, 128, 64), 'shifts': ((2, 3), (2, 3), (2,)), 'tau_mem_ms': 22.54, 'threshold': 0.5, 'split_seed': 12345}


ValueError: Checkpoint config mismatch for seed 11

## 7. Frozen representation extraction

For each checkpoint, run the continuous SNN once and record:

- L1 spike train: `[T,128]`
- L2 spike train: `[T,128]`
- L3 spike train: `[T,64]`
- L3 membrane trace: `[T,64]`

No gradient is enabled and the SNN weights are never changed.

In [ ]:
@torch.no_grad()
def extract_traces_for_array(model: ContinuousLocalFeatureSNN, X: np.ndarray):
    model = model.to(DEVICE)
    model.eval()

    all_l1 = []
    all_l2 = []
    all_l3 = []
    all_mem3 = []

    ds = TensorDataset(torch.from_numpy(X))
    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    for (xb,) in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        B, T, _ = xb.shape
        h1, h2, d = SNN_LAYER_WIDTHS

        syn1 = torch.zeros(B, h1, device=DEVICE)
        mem1 = torch.zeros_like(syn1)
        syn2 = torch.zeros(B, h2, device=DEVICE)
        mem2 = torch.zeros_like(syn2)
        syn3 = torch.zeros(B, d, device=DEVICE)
        mem3 = torch.zeros_like(syn3)

        l1_seq, l2_seq, l3_seq, mem3_seq = [], [], [], []

        for t in range(T):
            cur1 = model.fc1(xb[:, t])
            spk1, syn1, mem1 = model.lif1(cur1, syn1, mem1)

            cur2 = model.fc2(spk1)
            spk2, syn2, mem2 = model.lif2(cur2, syn2, mem2)

            cur3 = model.fc3(spk2)
            spk3, syn3, mem3 = model.lif3(cur3, syn3, mem3)

            l1_seq.append(spk1)
            l2_seq.append(spk2)
            l3_seq.append(spk3)
            mem3_seq.append(mem3)

        all_l1.append(torch.stack(l1_seq, dim=1).cpu().numpy())
        all_l2.append(torch.stack(l2_seq, dim=1).cpu().numpy())
        all_l3.append(torch.stack(l3_seq, dim=1).cpu().numpy())
        all_mem3.append(torch.stack(mem3_seq, dim=1).cpu().numpy())

    return {
        "l1_spikes": np.concatenate(all_l1, axis=0).astype(np.float32),
        "l2_spikes": np.concatenate(all_l2, axis=0).astype(np.float32),
        "l3_spikes": np.concatenate(all_l3, axis=0).astype(np.float32),
        "l3_mem": np.concatenate(all_mem3, axis=0).astype(np.float32),
    }


def firing_rate(trace: np.ndarray) -> float:
    return float(np.mean(trace))

## 8. Representation builders

In [9]:
def count_over_samples(trace: np.ndarray, samples_per_bin: int) -> np.ndarray:
    """[N,T,D] -> [N,T/bin,D] via non-overlapping sum."""
    N, T, D = trace.shape
    if T % samples_per_bin != 0:
        raise ValueError((T, samples_per_bin))
    return trace.reshape(N, T // samples_per_bin, samples_per_bin, D).sum(axis=2)


def mean_over_samples(trace: np.ndarray, samples_per_bin: int) -> np.ndarray:
    N, T, D = trace.shape
    if T % samples_per_bin != 0:
        raise ValueError((T, samples_per_bin))
    return trace.reshape(N, T // samples_per_bin, samples_per_bin, D).mean(axis=2)


def end_state_over_samples(trace: np.ndarray, samples_per_bin: int) -> np.ndarray:
    N, T, D = trace.shape
    if T % samples_per_bin != 0:
        raise ValueError((T, samples_per_bin))
    reshaped = trace.reshape(N, T // samples_per_bin, samples_per_bin, D)
    return reshaped[:, :, -1, :]


def raw_input_250ms_count(X: np.ndarray) -> np.ndarray:
    return count_over_samples(X, COARSE_BIN_SAMPLES)


def flatten_features(F: np.ndarray) -> np.ndarray:
    return np.asarray(F, dtype=np.float32).reshape(len(F), -1)


raw_train_250 = raw_input_250ms_count(X_train)
print("Raw 250 ms count shape:", raw_train_250.shape, "flattened:", flatten_features(raw_train_250).shape)

Raw 250 ms count shape: (632, 16, 30) flattened: (632, 480)


## 9. Fresh Logistic Regression probe protocol

In [10]:
def classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }


def fit_linear_probe(F_train, F_val, F_test, seed_tag: object):
    A = flatten_features(F_train)
    B = flatten_features(F_val)
    C = flatten_features(F_test)

    scaler = StandardScaler()
    A = scaler.fit_transform(A)
    B = scaler.transform(B)
    C = scaler.transform(C)

    clf = LogisticRegression(
        max_iter=LOGREG_MAX_ITER,
        solver="lbfgs",
        C=LOGREG_C,
        random_state=derive_seed(SPLIT_SEED, "probe", seed_tag),
    )
    clf.fit(A, y_train)

    return {
        "feature_dim": int(A.shape[1]),
        "train": classification_metrics(y_train, clf.predict(A)),
        "val": classification_metrics(y_val, clf.predict(B)),
        "test": classification_metrics(y_test, clf.predict(C)),
    }


# One seed-independent raw baseline.
raw_probe = fit_linear_probe(
    raw_input_250ms_count(X_train),
    raw_input_250ms_count(X_val),
    raw_input_250ms_count(X_test),
    seed_tag="raw_250ms_count",
)
raw_probe

{'feature_dim': 480,
 'train': {'accuracy': 1.0, 'balanced_accuracy': 1.0, 'macro_f1': 1.0},
 'val': {'accuracy': 0.7142857142857143,
  'balanced_accuracy': 0.7077050264550264,
  'macro_f1': 0.7066720517807474},
 'test': {'accuracy': 0.5473684210526316,
  'balanced_accuracy': 0.5562169312169313,
  'macro_f1': 0.548820612365504}}

## 10. Run all frozen probes

For each SNN seed, this cell extracts the train/val/test traces once and evaluates all three diagnostic groups.

In [12]:
probe_rows = []
firing_rows = []
representation_rows = []

# =====================================================================
# Raw input baseline
# =====================================================================

for split_name in ("train", "val", "test"):
    m = raw_probe[split_name]

    probe_rows.append(
        {
            "group": "A_layer_probe",
            "representation": "raw_input_250ms_count",
            "snn_seed": np.nan,
            "split": split_name,
            "feature_dim": raw_probe["feature_dim"],
            **m,
        }
    )


# =====================================================================
# Helper: extract one seed, one split at a time
# =====================================================================

def extract_all_split_traces(model):
    """
    Extract frozen SNN traces for train / val / test.

    Kept as a helper so the probe code below is easier to read.
    """

    print("  extracting train traces...")
    train_traces = extract_traces_for_array(
        model,
        X_train,
    )

    print("  extracting val traces...")
    val_traces = extract_traces_for_array(
        model,
        X_val,
    )

    print("  extracting test traces...")
    test_traces = extract_traces_for_array(
        model,
        X_test,
    )

    return {
        "train": train_traces,
        "val": val_traces,
        "test": test_traces,
    }


# =====================================================================
# Frozen-SNN probes
# =====================================================================

for seed in SNN_SEEDS:

    print("=" * 100)
    print("Frozen SNN seed:", seed)

    model, checkpoint = load_frozen_checkpoint(seed)

    model = model.to(DEVICE)
    model.eval()

    traces = extract_all_split_traces(model)

    # -----------------------------------------------------------------
    # Firing-rate diagnostic
    # -----------------------------------------------------------------

    for split_name, tr in traces.items():

        firing_rows.append(
            {
                "snn_seed": seed,
                "split": split_name,
                "l1_firing_rate": firing_rate(
                    tr["l1_spikes"]
                ),
                "l2_firing_rate": firing_rate(
                    tr["l2_spikes"]
                ),
                "l3_firing_rate": firing_rate(
                    tr["l3_spikes"]
                ),
            }
        )

    # =================================================================
    # A. Layer probe
    #
    # Same 250-ms spike-count readout for L1 / L2 / L3.
    # =================================================================

    layer_specs = {
        "l1_spike_250ms_count": "l1_spikes",
        "l2_spike_250ms_count": "l2_spikes",
        "l3_spike_250ms_count": "l3_spikes",
    }

    for rep_name, trace_key in layer_specs.items():

        print("  Layer probe:", rep_name)

        Ftr = count_over_samples(
            traces["train"][trace_key],
            COARSE_BIN_SAMPLES,
        )

        Fva = count_over_samples(
            traces["val"][trace_key],
            COARSE_BIN_SAMPLES,
        )

        Fte = count_over_samples(
            traces["test"][trace_key],
            COARSE_BIN_SAMPLES,
        )

        result = fit_linear_probe(
            Ftr,
            Fva,
            Fte,
            seed_tag=(seed, rep_name),
        )

        representation_rows.append(
            {
                "group": "A_layer_probe",
                "snn_seed": seed,
                "representation": rep_name,
                "shape_per_sample": tuple(
                    Ftr.shape[1:]
                ),
                "feature_dim": result[
                    "feature_dim"
                ],
            }
        )

        for split_name in (
            "train",
            "val",
            "test",
        ):

            probe_rows.append(
                {
                    "group": "A_layer_probe",
                    "representation": rep_name,
                    "snn_seed": seed,
                    "split": split_name,
                    "feature_dim": result[
                        "feature_dim"
                    ],
                    **result[split_name],
                }
            )

        del Ftr, Fva, Fte


    # =================================================================
    # B. Temporal-resolution probe
    #
    # Keep frozen L3 spikes.
    # Only change output temporal aggregation:
    #
    #   16 samples = 250 ms
    #    8 samples = 125 ms
    #    4 samples = 62.5 ms
    # =================================================================

    for samples_per_bin in TEMPORAL_READOUT_SAMPLES:

        duration_ms = (
            1000.0
            * samples_per_bin
            / SAMPLING_RATE_HZ
        )

        rep_name = (
            f"l3_spike_"
            f"{duration_ms:g}ms_count"
        )

        print(
            "  Temporal-resolution probe:",
            rep_name,
        )

        Ftr = count_over_samples(
            traces["train"]["l3_spikes"],
            samples_per_bin,
        )

        Fva = count_over_samples(
            traces["val"]["l3_spikes"],
            samples_per_bin,
        )

        Fte = count_over_samples(
            traces["test"]["l3_spikes"],
            samples_per_bin,
        )

        result = fit_linear_probe(
            Ftr,
            Fva,
            Fte,
            seed_tag=(
                seed,
                rep_name,
                "temporal",
            ),
        )

        representation_rows.append(
            {
                "group": "B_temporal_resolution",
                "snn_seed": seed,
                "representation": rep_name,
                "shape_per_sample": tuple(
                    Ftr.shape[1:]
                ),
                "feature_dim": result[
                    "feature_dim"
                ],
            }
        )

        for split_name in (
            "train",
            "val",
            "test",
        ):

            probe_rows.append(
                {
                    "group": "B_temporal_resolution",
                    "representation": rep_name,
                    "snn_seed": seed,
                    "split": split_name,
                    "feature_dim": result[
                        "feature_dim"
                    ],
                    **result[split_name],
                }
            )

        del Ftr, Fva, Fte


    # =================================================================
    # C. State-readout probe
    #
    # Same 250-ms coarse positions, but compare:
    #
    #   spike count
    #   membrane end state
    #   membrane mean
    #
    # Note:
    # l3_spike_250ms_count was already evaluated above, but it is
    # intentionally added to group C again so the C-state plot has
    # its own spike baseline.
    # =================================================================

    state_specs = {
        "l3_spike_250ms_count": (
            "l3_spikes",
            lambda x: count_over_samples(
                x,
                COARSE_BIN_SAMPLES,
            ),
        ),

        "l3_mem_250ms_end": (
            "l3_mem",
            lambda x: end_state_over_samples(
                x,
                COARSE_BIN_SAMPLES,
            ),
        ),

        "l3_mem_250ms_mean": (
            "l3_mem",
            lambda x: mean_over_samples(
                x,
                COARSE_BIN_SAMPLES,
            ),
        ),
    }

    for (
        rep_name,
        (trace_key, builder),
    ) in state_specs.items():

        print(
            "  State-readout probe:",
            rep_name,
        )

        Ftr = builder(
            traces["train"][trace_key]
        )

        Fva = builder(
            traces["val"][trace_key]
        )

        Fte = builder(
            traces["test"][trace_key]
        )

        result = fit_linear_probe(
            Ftr,
            Fva,
            Fte,
            seed_tag=(
                seed,
                rep_name,
                "state",
            ),
        )

        representation_rows.append(
            {
                "group": "C_state_readout",
                "snn_seed": seed,
                "representation": rep_name,
                "shape_per_sample": tuple(
                    Ftr.shape[1:]
                ),
                "feature_dim": result[
                    "feature_dim"
                ],
            }
        )

        for split_name in (
            "train",
            "val",
            "test",
        ):

            probe_rows.append(
                {
                    "group": "C_state_readout",
                    "representation": rep_name,
                    "snn_seed": seed,
                    "split": split_name,
                    "feature_dim": result[
                        "feature_dim"
                    ],
                    **result[split_name],
                }
            )

        del Ftr, Fva, Fte


    # -----------------------------------------------------------------
    # Release large traces before loading the next seed.
    # -----------------------------------------------------------------

    del traces
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =====================================================================
# Final tables
# =====================================================================

probe_df = pd.DataFrame(
    probe_rows
)

firing_df = pd.DataFrame(
    firing_rows
)

representation_df = (
    pd.DataFrame(
        representation_rows
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print()
print(
    "Probe rows:",
    len(probe_df),
)

print()
print("Firing rates:")
display(
    firing_df
)

print()
print("Representation shapes:")
display(
    representation_df
)

Frozen SNN seed: 11
Expected config: {'experiment_id': 'experiment_1_3_5_continuous_snn_local_features', 'sampling_rate_hz': 64.0, 'event_channels': 30, 'global_padded_length': 256, 'padded_readout_length': 256, 'bin_samples': 16, 'fixed_bin_ms': 250.0, 'layer_widths': (128, 128, 64), 'layer_shifts': ((2, 3), (2, 3), (2, 3)), 'tau_mem_ms': 22.54, 'threshold': 0.5, 'split_seed': 12345}
Checkpoint config: {'experiment_id': 'experiment_1_3_5_continuous_snn_local_features', 'fs': 64.0, 'bin_samples': 16, 'widths': (128, 128, 64), 'shifts': ((2, 3), (2, 3), (2,)), 'tau_mem_ms': 22.54, 'threshold': 0.5, 'split_seed': 12345}


ValueError: Checkpoint config mismatch for seed 11

## 11. Group A — Layer probe results

In [ ]:
layer_test = probe_df.query("group == 'A_layer_probe' and split == 'test'").copy()

# Seed-independent raw baseline + mean/sd over SNN seeds.
raw_row = layer_test[layer_test.representation == "raw_input_250ms_count"].iloc[0]
layer_summary_rows = [{
    "representation": "raw_input_250ms_count",
    "n_runs": 1,
    "mean_test_BA": raw_row.balanced_accuracy,
    "sd_test_BA": np.nan,
    "mean_test_accuracy": raw_row.accuracy,
    "mean_test_macro_f1": raw_row.macro_f1,
    "feature_dim": int(raw_row.feature_dim),
}]

for rep_name, g in layer_test[layer_test.representation != "raw_input_250ms_count"].groupby("representation"):
    layer_summary_rows.append({
        "representation": rep_name,
        "n_runs": len(g),
        "mean_test_BA": g.balanced_accuracy.mean(),
        "sd_test_BA": g.balanced_accuracy.std(ddof=1),
        "mean_test_accuracy": g.accuracy.mean(),
        "mean_test_macro_f1": g.macro_f1.mean(),
        "feature_dim": int(g.feature_dim.iloc[0]),
    })

layer_summary = pd.DataFrame(layer_summary_rows)
display(layer_summary)

plt.figure(figsize=(9, 5))
plot_df = layer_summary.copy()
x = np.arange(len(plot_df))
err = plot_df.sd_test_BA.fillna(0).to_numpy()
plt.bar(x, plot_df.mean_test_BA, yerr=err, capsize=4)
plt.xticks(x, ["Raw input", "L1 spikes", "L2 spikes", "L3 spikes"], rotation=10)
plt.ylabel("Test balanced accuracy")
plt.title("1.3.6A — Layer probe (250 ms count + Linear)")
plt.grid(axis="y", alpha=0.25)
plt.show()

## 12. Group B — Temporal-resolution probe results

In [ ]:
temporal_test = probe_df.query("group == 'B_temporal_resolution' and split == 'test'").copy()

temporal_summary = (
    temporal_test.groupby("representation", as_index=False)
    .agg(
        n_runs=("snn_seed", "count"),
        mean_test_BA=("balanced_accuracy", "mean"),
        sd_test_BA=("balanced_accuracy", "std"),
        mean_test_accuracy=("accuracy", "mean"),
        mean_test_macro_f1=("macro_f1", "mean"),
        feature_dim=("feature_dim", "first"),
    )
)

order = [
    "l3_spike_250ms_count",
    "l3_spike_125ms_count",
    "l3_spike_62.5ms_count",
]
temporal_summary["order"] = temporal_summary.representation.map({k:i for i,k in enumerate(order)})
temporal_summary = temporal_summary.sort_values("order").drop(columns="order")
display(temporal_summary)

plt.figure(figsize=(8, 5))
x = np.arange(len(temporal_summary))
plt.bar(
    x,
    temporal_summary.mean_test_BA,
    yerr=temporal_summary.sd_test_BA,
    capsize=4,
)
plt.xticks(x, ["250 ms", "125 ms", "62.5 ms"])
plt.ylabel("Test balanced accuracy")
plt.title("1.3.6B — L3 spike readout temporal resolution")
plt.grid(axis="y", alpha=0.25)
plt.show()

## 13. Group C — State-readout probe results

In [ ]:
state_test = probe_df.query("group == 'C_state_readout' and split == 'test'").copy()

state_summary = (
    state_test.groupby("representation", as_index=False)
    .agg(
        n_runs=("snn_seed", "count"),
        mean_test_BA=("balanced_accuracy", "mean"),
        sd_test_BA=("balanced_accuracy", "std"),
        mean_test_accuracy=("accuracy", "mean"),
        mean_test_macro_f1=("macro_f1", "mean"),
        feature_dim=("feature_dim", "first"),
    )
)

order = [
    "l3_spike_250ms_count",
    "l3_mem_250ms_end",
    "l3_mem_250ms_mean",
]
state_summary["order"] = state_summary.representation.map({k:i for i,k in enumerate(order)})
state_summary = state_summary.sort_values("order").drop(columns="order")
display(state_summary)

plt.figure(figsize=(8, 5))
x = np.arange(len(state_summary))
plt.bar(
    x,
    state_summary.mean_test_BA,
    yerr=state_summary.sd_test_BA,
    capsize=4,
)
plt.xticks(x, ["Spike count", "Mem end", "Mem mean"])
plt.ylabel("Test balanced accuracy")
plt.title("1.3.6C — L3 readout type")
plt.grid(axis="y", alpha=0.25)
plt.show()

## 14. Per-seed detailed test table

In [ ]:
test_detail = probe_df[probe_df.split == "test"].sort_values(
    ["group", "representation", "snn_seed"],
    na_position="first",
).reset_index(drop=True)

display(test_detail)

## 15. Firing-rate diagnostics

In [ ]:
display(firing_df)

firing_summary = (
    firing_df.groupby("split", as_index=False)
    .agg(
        l1_fr_mean=("l1_firing_rate", "mean"),
        l1_fr_sd=("l1_firing_rate", "std"),
        l2_fr_mean=("l2_firing_rate", "mean"),
        l2_fr_sd=("l2_firing_rate", "std"),
        l3_fr_mean=("l3_firing_rate", "mean"),
        l3_fr_sd=("l3_firing_rate", "std"),
    )
)
display(firing_summary)

## 16. Interpretation / decision rules

Use the result pattern to decide the next architecture change.

### Case 1 — L1 is strongest, deeper layers degrade

Example pattern:

\[
Raw \lesssim L1 > L2 > L3
\]

Interpretation: the SNN may extract a useful shallow local representation, while deeper spike-to-spike transformations over-compress or overfit it. The next experiment should reduce depth rather than increase width.

### Case 2 — finer L3 temporal readout improves performance

\[
L3_{62.5ms} > L3_{125ms} > L3_{250ms}
\]

Interpretation: useful timing exists in the L3 spike train, but the original 250 ms output count discarded it. Keep the SNN and change the readout resolution before changing the backbone.

### Case 3 — membrane readout beats spike count

\[
L3_{mem} > L3_{spike-count}
\]

Interpretation: continuous SNN state contains useful information that is lost by thresholding/counting. Investigate membrane/state readout or a firing-regime change.

### Case 4 — every frozen SNN representation is below raw count

Interpretation: the current SNN transformation itself is reducing cross-user robustness. Do not add more capacity. A strong next direction is a residual two-branch representation:

\[
\boxed{
\text{robust raw count feature}
+
\text{SNN fine-temporal residual feature}
}
\]

rather than asking the SNN to replace the count representation completely.

## 17. Save tables

In [ ]:
probe_df.to_csv(RESULTS_DIR / "all_probe_results.csv", index=False)
firing_df.to_csv(RESULTS_DIR / "firing_rates.csv", index=False)
representation_df.to_csv(RESULTS_DIR / "representation_dimensions.csv", index=False)
layer_summary.to_csv(RESULTS_DIR / "layer_probe_summary.csv", index=False)
temporal_summary.to_csv(RESULTS_DIR / "temporal_resolution_summary.csv", index=False)
state_summary.to_csv(RESULTS_DIR / "state_readout_summary.csv", index=False)

print("Saved Experiment 1.3.6 results to:", RESULTS_DIR)